In [11]:
####################################
# imports and global configuration
####################################
import json
import math
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm

# These settings control how much work is saved at a time. Smaller values save
# more often, which is safer for long Colab runs but slightly slower.
BEAM_WIDTH = 5
SOFTMAX_THRESHOLD = 0.90
SAVE_EVERY_ROWS = 100
MAX_ROWS = None  # Set to a small integer for testing. Use None for the full file.

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cpu


In [12]:
####################################
# file paths
####################################
# In Colab, this uses the arabizi folder at the top of MyDrive. Locally, it
# falls back to the project files so the notebook can still be tested.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ARABIZI_DIR = Path("/content/drive/MyDrive/arabizi")
    DATA_FILE = ARABIZI_DIR / "maknuune-v1.0.1_cleaned.csv"
    MODEL_FILE = ARABIZI_DIR / "harkat_adder.pt"
except ModuleNotFoundError:
    PROJECT_DIR = Path.cwd()
    if PROJECT_DIR.name == "train_harakat_adder":
        PROJECT_DIR = PROJECT_DIR.parents[1]
    DATA_FILE = PROJECT_DIR /"maknuune-v1.0.1_cleaned.csv"
    MODEL_FILE = PROJECT_DIR / "harkat_adder.pt"
    ARABIZI_DIR = DATA_FILE.parent

PREDICTIONS_FILE = ARABIZI_DIR / "maknuune-v1.0.1_cleaned_predictions.csv"
METRICS_FILE = ARABIZI_DIR / "harakat_prediction_metrics_by_length.csv"

print("Arabizi directory:", ARABIZI_DIR)
print("Data file:", DATA_FILE)
print("Model file:", MODEL_FILE)
print("Predictions file:", PREDICTIONS_FILE)
print("Metrics file:", METRICS_FILE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Arabizi directory: /content/drive/MyDrive/arabizi
Data file: /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned.csv
Model file: /content/drive/MyDrive/arabizi/harkat_adder.pt
Predictions file: /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_predictions.csv
Metrics file: /content/drive/MyDrive/arabizi/harakat_prediction_metrics_by_length.csv


In [ ]:
####################################
# load the saved model bundle
####################################
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Could not find cleaned data file: {DATA_FILE}")
if not MODEL_FILE.exists():
    raise FileNotFoundError(f"Could not find trained model file: {MODEL_FILE}")

# The bundle saved by train_harakat_adder.ipynb contains both the model weights
# and the exact vocabularies/configuration needed to rebuild the architecture.
bundle = torch.load(MODEL_FILE, map_location=DEVICE)
MODEL_CONFIG = bundle["config"]
SRC_STOI = bundle["src_stoi"]
SRC_ITOS = bundle["src_itos"]
TGT_STOI = bundle["tgt_stoi"]
TGT_ITOS = bundle["tgt_itos"]

PAD = "<PAD>"
BOS = "<BOS>"
EOS = "<EOS>"
UNK = "<UNK>"
SPECIAL_TOKENS = [PAD, BOS, EOS, UNK]

D_MODEL = MODEL_CONFIG["d_model"]
NHEAD = MODEL_CONFIG["nhead"]
NUM_ENCODER_LAYERS = MODEL_CONFIG["num_encoder_layers"]
NUM_DECODER_LAYERS = MODEL_CONFIG["num_decoder_layers"]
DIM_FEEDFORWARD = MODEL_CONFIG["dim_feedforward"]
DROPOUT = MODEL_CONFIG["dropout"]
MAX_LEN = MODEL_CONFIG["max_len"]
SRC_PAD_IDX = MODEL_CONFIG["src_pad_idx"]
TGT_PAD_IDX = MODEL_CONFIG["tgt_pad_idx"]

print("Loaded model bundle saved at:", bundle.get("saved_at", "unknown"))
print("Model config:", MODEL_CONFIG)

In [ ]:
####################################
# basic text and vocabulary helpers
####################################
def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")


def clean_text(value: object) -> str:
    if pd.isna(value):
        return ""
    return " ".join(str(value).split())


def encode_text(text: str, stoi: dict[str, int], add_bos: bool = False, add_eos: bool = True) -> list[int]:
    ids = []
    if add_bos:
        ids.append(stoi[BOS])
    ids.extend(stoi.get(ch, stoi[UNK]) for ch in text)
    if add_eos:
        ids.append(stoi[EOS])
    return ids


def decode_ids(ids: list[int], itos: dict[int, str]) -> str:
    chars = []
    for idx in ids:
        token = itos.get(int(idx), UNK)
        if token == EOS:
            break
        if token not in SPECIAL_TOKENS:
            chars.append(token)
    return "".join(chars)

In [15]:
####################################
# model definition
####################################
# This is the same encoder-decoder Transformer used during training. The saved
# bundle supplies the learned weights and vocabulary IDs.
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float, max_len: int):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)


class HarakatAdderTransformer(nn.Module):
    def __init__(self, src_vocab_size: int, tgt_vocab_size: int, max_len: int, src_pad_idx: int, tgt_pad_idx: int):
        super().__init__()
        self.src_pad_idx = src_pad_idx
        self.tgt_pad_idx = tgt_pad_idx
        self.d_model = D_MODEL

        self.src_embedding = nn.Embedding(src_vocab_size, D_MODEL, padding_idx=src_pad_idx)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, D_MODEL, padding_idx=tgt_pad_idx)
        self.positional_encoding = PositionalEncoding(D_MODEL, DROPOUT, max_len=max_len)

        self.transformer = nn.Transformer(
            d_model=D_MODEL,
            nhead=NHEAD,
            num_encoder_layers=NUM_ENCODER_LAYERS,
            num_decoder_layers=NUM_DECODER_LAYERS,
            dim_feedforward=DIM_FEEDFORWARD,
            dropout=DROPOUT,
            batch_first=True,
        )
        self.output_layer = nn.Linear(D_MODEL, tgt_vocab_size)

    def forward(self, src: torch.Tensor, tgt_input: torch.Tensor) -> torch.Tensor:
        src_padding_mask = src.eq(self.src_pad_idx)
        tgt_padding_mask = tgt_input.eq(self.tgt_pad_idx)

        tgt_len = tgt_input.size(1)
        tgt_mask = torch.triu(
            torch.ones(tgt_len, tgt_len, device=tgt_input.device, dtype=torch.bool),
            diagonal=1,
        )

        src_emb = self.positional_encoding(self.src_embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.positional_encoding(self.tgt_embedding(tgt_input) * math.sqrt(self.d_model))

        hidden = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask,
        )
        return self.output_layer(hidden)


model = HarakatAdderTransformer(
    src_vocab_size=MODEL_CONFIG["src_vocab_size"],
    tgt_vocab_size=MODEL_CONFIG["tgt_vocab_size"],
    max_len=MAX_LEN,
    src_pad_idx=SRC_PAD_IDX,
    tgt_pad_idx=TGT_PAD_IDX,
).to(DEVICE)

model.load_state_dict(bundle["model_state_dict"])
model.eval()
print("Model loaded and ready for prediction.")

Model loaded and ready for prediction.


In [16]:
####################################
# threshold-based beam search prediction
####################################
# At each character step, this expands enough next-token choices to cover 90%
# of the softmax probability mass, while keeping only the best beam paths.
@torch.no_grad()
def beam_search_decode_threshold(model, src_sentence: str, beam_width: int = BEAM_WIDTH, threshold: float = SOFTMAX_THRESHOLD, max_len: int = MAX_LEN) -> list[dict]:
    model.eval()
    src_ids = encode_text(src_sentence, SRC_STOI, add_bos=False, add_eos=True)
    src_tensor = torch.tensor([src_ids], dtype=torch.long, device=DEVICE)

    beams = [([TGT_STOI[BOS]], 0.0)]
    completed_beams = []

    src_padding_mask = src_tensor.eq(SRC_PAD_IDX)
    src_emb = model.positional_encoding(model.src_embedding(src_tensor) * math.sqrt(model.d_model))
    memory = model.transformer.encoder(src_emb, src_key_padding_mask=src_padding_mask)

    for _ in range(max_len):
        new_beams = []
        for seq, score in beams:
            if seq[-1] == TGT_STOI[EOS]:
                completed_beams.append((seq, score))
                continue

            tgt_input = torch.tensor([seq], dtype=torch.long, device=DEVICE)
            tgt_mask = torch.triu(torch.ones(len(seq), len(seq), device=DEVICE, dtype=torch.bool), diagonal=1)

            tgt_emb = model.positional_encoding(model.tgt_embedding(tgt_input) * math.sqrt(model.d_model))
            output = model.transformer.decoder(tgt_emb, memory, tgt_mask=tgt_mask, memory_key_padding_mask=src_padding_mask)
            logits = model.output_layer(output[:, -1, :])

            probs = torch.softmax(logits, dim=-1).squeeze(0)
            sorted_probs, sorted_indices = torch.sort(probs, descending=True)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff_idx = torch.where(cumulative_probs >= threshold)[0][0].item() + 1
            num_to_take = min(max(beam_width, cutoff_idx), 10)

            for i in range(num_to_take):
                token_probability = max(float(sorted_probs[i].item()), 1e-45)
                new_seq = seq + [int(sorted_indices[i].item())]
                new_score = score + math.log(token_probability)
                new_beams.append((new_seq, new_score))

        if not new_beams:
            break
        beams = sorted(new_beams, key=lambda item: item[1], reverse=True)[:beam_width]

    all_candidates = sorted(completed_beams + beams, key=lambda item: item[1], reverse=True)[:beam_width]
    predictions = []
    for seq, log_probability in all_candidates:
        text = decode_ids(seq, TGT_ITOS).strip()
        predictions.append({
            "text": text,
            "probability": math.exp(log_probability) if log_probability > -745 else 0.0,
            "log_probability": log_probability,
        })
    return predictions

In [17]:
####################################
# load data and resume saved predictions
####################################
PREDICTION_COLUMNS = [
    "harakat_predictions_softmax_90",
    "harakat_top_prediction",
    "harakat_top_probability",
    "harakat_top_log_probability",
    "harakat_prediction_done_at",
]

if PREDICTIONS_FILE.exists():
    df = pd.read_csv(PREDICTIONS_FILE)
    print("Resuming from existing predictions file.")
else:
    df = pd.read_csv(DATA_FILE)
    print("Starting from the cleaned data file.")

required_columns = {"arabic_stripped", "arabic_harakat"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns in {DATA_FILE}: {sorted(missing_columns)}")

df["arabic_stripped"] = df["arabic_stripped"].map(clean_text)
df["arabic_harakat"] = df["arabic_harakat"].map(clean_text)

for column in PREDICTION_COLUMNS:
    if column not in df.columns:
        df[column] = pd.NA

if MAX_ROWS is not None:
    df = df.head(MAX_ROWS).copy()

completed_mask = df["harakat_predictions_softmax_90"].notna() & (df["harakat_predictions_softmax_90"].astype(str).str.len() > 0)
print("Rows:", len(df))
print("Already completed:", int(completed_mask.sum()))
print("Remaining:", int((~completed_mask).sum()))

Resuming from existing predictions file.
Rows: 36302
Already completed: 36302
Remaining: 0


In [10]:
####################################
# run predictions with periodic saving
####################################
# This loop is intentionally resumable. If Colab disconnects, rerun the cells
# above and this cell will skip rows that already have saved predictions.
rows_since_save = 0
pending_indices = df.index[~completed_mask].tolist()

for row_idx in tqdm(pending_indices, desc="Predicting harakat"):
    source_text = clean_text(df.at[row_idx, "arabic_stripped"])
    predictions = beam_search_decode_threshold(model, source_text)

    df.at[row_idx, "harakat_predictions_softmax_90"] = json.dumps(predictions, ensure_ascii=False)
    if predictions:
        df.at[row_idx, "harakat_top_prediction"] = predictions[0]["text"]
        df.at[row_idx, "harakat_top_probability"] = predictions[0]["probability"]
        df.at[row_idx, "harakat_top_log_probability"] = predictions[0]["log_probability"]
    df.at[row_idx, "harakat_prediction_done_at"] = now_iso()

    rows_since_save += 1
    if rows_since_save >= SAVE_EVERY_ROWS:
        df.to_csv(PREDICTIONS_FILE, index=False)
        print(f"Saved progress through row {row_idx} to {PREDICTIONS_FILE}")
        rows_since_save = 0

df.to_csv(PREDICTIONS_FILE, index=False)
print("Prediction pass complete.")
print("Saved predictions to:", PREDICTIONS_FILE)

Predicting harakat:   0%|          | 0/36302 [00:00<?, ?it/s]

Saved progress through row 99 to /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_predictions.csv
Saved progress through row 199 to /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_predictions.csv
Saved progress through row 299 to /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_predictions.csv
Saved progress through row 399 to /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_predictions.csv
Saved progress through row 499 to /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_predictions.csv
Saved progress through row 599 to /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_predictions.csv
Saved progress through row 699 to /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_predictions.csv
Saved progress through row 799 to /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_predictions.csv
Saved progress through row 899 to /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_predictions.csv
Saved progress through row 999 to /content/drive/MyDrive

In [18]:
####################################
# accuracy metrics
####################################
def target_in_prediction_json(prediction_json: object, target_text: str) -> bool:
    if pd.isna(prediction_json):
        return False
    try:
        predictions = json.loads(prediction_json)
    except json.JSONDecodeError:
        return False
    return clean_text(target_text) in {clean_text(item.get("text", "")) for item in predictions}


completed_for_metrics = df["harakat_predictions_softmax_90"].notna() & (df["harakat_predictions_softmax_90"].astype(str).str.len() > 0)
metrics_df = df.loc[completed_for_metrics].copy()
if metrics_df.empty:
    raise ValueError("No completed predictions found yet. Run the prediction cell before calculating metrics.")

metrics_df["arabic_stripped_length"] = metrics_df["arabic_stripped"].map(len)
metrics_df["length_bucket"] = (metrics_df["arabic_stripped_length"] // 10 * 10).map(lambda start: f"{start}-{start + 9}")
metrics_df["top1_exact_match"] = metrics_df["harakat_top_prediction"].map(clean_text) == metrics_df["arabic_harakat"].map(clean_text)
metrics_df["target_in_softmax_90_list"] = metrics_df.apply(
    lambda row: target_in_prediction_json(row["harakat_predictions_softmax_90"], row["arabic_harakat"]),
    axis=1,
)

overall_metrics = pd.DataFrame([
    {
        "group": "overall",
        "count": len(metrics_df),
        "top1_exact_accuracy": metrics_df["top1_exact_match"].mean(),
        "target_in_softmax_90_accuracy": metrics_df["target_in_softmax_90_list"].mean(),
        "avg_source_length": metrics_df["arabic_stripped_length"].mean(),
    }
])

length_metrics = (
    metrics_df.groupby("length_bucket", sort=False)
    .agg(
        count=("arabic_stripped", "size"),
        top1_exact_accuracy=("top1_exact_match", "mean"),
        target_in_softmax_90_accuracy=("target_in_softmax_90_list", "mean"),
        avg_source_length=("arabic_stripped_length", "mean"),
    )
    .reset_index()
    .rename(columns={"length_bucket": "group"})
)

all_metrics = pd.concat([overall_metrics, length_metrics], ignore_index=True)
all_metrics.to_csv(METRICS_FILE, index=False)

display(overall_metrics)
display(length_metrics.sort_values("group"))
print("Saved metrics to:", METRICS_FILE)

,group,count,top1_exact_accuracy,target_in_softmax_90_accuracy,avg_source_length
0,overall,36302,0.645254,0.949893,5.066801


,group,count,top1_exact_accuracy,target_in_softmax_90_accuracy,avg_source_length
0,0-9,34072,0.648773,0.961082,4.421196
1,10-19,1888,0.618114,0.802966,12.676377
2,20-29,255,0.482353,0.705882,23.556863
3,30-39,64,0.359375,0.531250,33.750000
4,40-49,11,0.363636,0.363636,45.545455
5,50-59,8,0.250000,0.375000,51.625000
6,60-69,2,0.000000,0.000000,66.000000
7,70-79,2,0.000000,0.000000,75.000000


Saved metrics to: /content/drive/MyDrive/arabizi/harakat_prediction_metrics_by_length.csv


In [19]:
####################################
# preview saved predictions
####################################
preview_columns = [
    "arabic_stripped",
    "arabic_harakat",
    "harakat_top_prediction",
    "harakat_top_probability",
    "harakat_predictions_softmax_90",
]
display(df[preview_columns].head(10))

,arabic_stripped,arabic_harakat,harakat_top_prediction,harakat_top_probability,harakat_predictions_softmax_90
0,أبد,أَبَد,أَبَد,0.432208,"[{""text"": ""أَبَد"", ""probability"": 0.4322081426..."
1,إبرة,إِبْرِة,إِبْرَة,0.784479,"[{""text"": ""إِبْرَة"", ""probability"": 0.78447850..."
2,إبر,إِبَر,إِبِر,0.709194,"[{""text"": ""إِبِر"", ""probability"": 0.7091937224..."
3,قد خرم الإبرة,قَدّ خُرُم الإِبْرِة,قَدّ خُرُم الإِبْرِة,0.589763,"[{""text"": ""قَدّ خُرُم الإِبْرِة"", ""probability..."
4,إبرة العجوزة,إِبْرِة العَجُوزِة,إِبْرِة العَجُوزِة,0.369413,"[{""text"": ""إِبْرِة العَجُوزِة"", ""probability"":..."
5,الإبرة غلبت الحايك,الإِبْرِة غلبت الحَايِك,الإِبْرِة غلبت الحَايِك,0.221884,"[{""text"": ""الإِبْرِة غلبت الحَايِك"", ""probabil..."
6,من إبرته,مِن إِبْرِتُه,مِن إِبْرِتُه,0.476769,"[{""text"": ""مِن إِبْرِتُه"", ""probability"": 0.47..."
7,أباط,أَبَاط,أَبَاط,0.883475,"[{""text"": ""أَبَاط"", ""probability"": 0.883474513..."
8,باط,بَاط,بَاط,0.982089,"[{""text"": ""بَاط"", ""probability"": 0.98208885705..."
9,حطه تحت باطه,حَطُّه تحت بَاطُه,حَطُّه تَحْت بَطَاه,0.081647,"[{""text"": ""حَطُّه تَحْت بَطَاه"", ""probability""..."
